# Suburban Infrastructure Poverty Map

**Project:** Suburban Futures (MDAP, University of Melbourne)
**Purpose:** Visualize spatial mismatches between population growth and public infrastructure across Greater Melbourne  
**Output:** Tidy GeoJSON files for Kepler.gl visualization  
**CRS:** GDA2020 (EPSG:7844)

---

## Setup Instructions

Before running this notebook, make sure the repo uses the expected project structure:

1. **`data/raw/population/population.gpkg`** — ABS Estimated Resident Population (ERP) for SA2 areas (GDA2020).
2. **`data/raw/vicmap/Order_PA8H1X/.../VMFEAT/FOI_POINT.*`** — Vicmap FOI shapefile components including `FOI_POINT.shp`, `FOI_POINT.shx`, `FOI_POINT.dbf`, and `FOI_POINT.prj`. 
3. **`data/processed/melbourne.gpkg`** — Greater Melbourne boundary package used for filtering and joins.

The configuration cell below resolves these files automatically from the repo root and writes exports into `outputs/`.

---

## Data Sources

| Dataset | Source | Format | Geographic Level |
|---|---|---|---|
| Estimated Resident Population | Australian Bureau of Statistics (ABS) | GeoPackage (.gpkg) | SA2 (Statistical Area Level 2) |
| Vicmap Features of Interest | Department of Transport and Planning (Victoria) | Shapefile (.shp) | Individual point locations |

## Configuration and Imports

Update the file paths below to match your local setup.

### Set File Paths

This cell defines the two input datasets used throughout the notebook.  
Only the infrastructure shapefile path normally needs to be updated.

In [ ]:
# =============================================================
# FILE PATHS — repo-relative project locations
# =============================================================
from pathlib import Path

CWD = Path.cwd().resolve()
if (CWD / "data").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "data").exists():
    REPO_ROOT = CWD.parent
else:
    raise FileNotFoundError("Could not find the repo root containing the data/ directory.")

DATA_RAW_DIR = REPO_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = REPO_ROOT / "data" / "processed"
OUTPUTS_DIR = REPO_ROOT / "outputs"
OUTPUT_GEOJSON_DIR = OUTPUTS_DIR / "geojson"
OUTPUT_TABLES_DIR = OUTPUTS_DIR / "tables"

for path in [OUTPUT_GEOJSON_DIR, OUTPUT_TABLES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

POPULATION_PATH = DATA_RAW_DIR / "population" / "population.gpkg"
INFRA_PATH = DATA_RAW_DIR / "vicmap" / "Order_PA8H1X" / "ll_gda2020" / "esrishape" / "whole_of_dataset" / "victoria" / "VMFEAT" / "FOI_POINT.shp"
MELBOURNE_PATH = DATA_PROCESSED_DIR / "melbourne.gpkg"

POPULATION_GROWTH_OUTPUT = OUTPUT_GEOJSON_DIR / "kepler_population_growth.geojson"
INFRA_POINTS_OUTPUT = OUTPUT_GEOJSON_DIR / "kepler_infrastructure_points.geojson"
POPULATION_TS_OUTPUT = OUTPUT_GEOJSON_DIR / "kepler_population_timeseries.geojson"
INFRA_TS_OUTPUT = OUTPUT_GEOJSON_DIR / "kepler_infrastructure_timeseries.geojson"
SUMMARY_OUTPUT = OUTPUT_TABLES_DIR / "infrastructure_poverty_summary.csv"

# =============================================================
# No changes needed below this line
# =============================================================

### Import Libraries

We load the geospatial, table, numeric, and plotting libraries used in the  
rest of the workflow.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load Data and Verify CRS Alignment

Both datasets must be in the same Coordinate Reference System (GDA2020 / EPSG:7844)  
for the spatial operations to work correctly.

In [ ]:
# Load both datasets
pop_gdf = gpd.read_file(POPULATION_PATH, layer='ERP_SA2_GDA2020')
infra_gdf = gpd.read_file(INFRA_PATH)

# Verify CRS alignment
print("=== CRS CHECK ===")
print(f"Population:     {pop_gdf.crs.name} (EPSG:{pop_gdf.crs.to_epsg()})")
print(f"Infrastructure: {infra_gdf.crs.name} (EPSG:{infra_gdf.crs.to_epsg()})")

if pop_gdf.crs.to_epsg() == infra_gdf.crs.to_epsg():
    print("Both files are in the SAME CRS. No offset expected.")
else:
    print(f"DIFFERENT CRS — will be aligned to EPSG:7844 during processing.")

print(f"\nPopulation rows: {len(pop_gdf)}")
print(f"Infrastructure rows: {len(infra_gdf)}")

## Filter Population Data to Greater Melbourne

The ABS data covers all of Victoria. We filter to only Greater Melbourne  
using the `gccsa_name_2021` column, then create a dissolved boundary  
for spatially clipping infrastructure points later.

In [ ]:
# Filter to Greater Melbourne SA2 areas
melb_pop = pop_gdf[pop_gdf['gccsa_name_2021'] == 'Greater Melbourne'].copy()
melb_pop = melb_pop.to_crs(epsg=7844)

# Dissolve all SA2 polygons into one unified Melbourne boundary.
# This eliminates micro-gaps between individual SA2 polygon edges
# that caused infrastructure points to leak outside the boundary
# when using gpd.sjoin() against individual polygons.
melb_boundary = melb_pop.dissolve()

print(f"Melbourne SA2 areas: {len(melb_pop)}")
print(f"Boundary type: {melb_boundary.geometry.iloc[0].geom_type}")

### Inspect LGA Layer

Before joining council names to SA2 areas, we quickly inspect the LGA layer  
to confirm the correct field name and geometry structure.

In [ ]:
# Inspect LGA layer to find the name column
lga = gpd.read_file(MELBOURNE_PATH, layer='greater_melb_local_gov_2025_any_intersect')

print("Columns:")
print(lga.columns.tolist())
print("\nFirst 2 rows:")
lga.head(2)

### Attach Council Names To SA2 Areas

We assign each SA2 to a 2025 local government area using a centroid-in-polygon  
join. This ensures council names are carried forward into both Current Infrastructure Poverty and Infrastructure Poverty Timeline  
exports.

In [ ]:
# Attach LGA (council) name to each SA2 via centroid-in-polygon spatial join
lga = gpd.read_file(MELBOURNE_PATH, layer='greater_melb_local_gov_2025_any_intersect')
lga = lga[['LGA_NAME_2025', 'geometry']]
lga = lga.to_crs(melb_pop.crs)

# Centroids in a projected CRS (GDA2020 / MGA zone 55) for accuracy
sa2_centroids = melb_pop.to_crs(7855).copy()
sa2_centroids['geometry'] = sa2_centroids.geometry.centroid
sa2_centroids = sa2_centroids.to_crs(melb_pop.crs)

joined = gpd.sjoin(
    sa2_centroids[['sa2_code_2021', 'geometry']],
    lga,
    how='left',
    predicate='within'
)

melb_pop = melb_pop.merge(
    joined[['sa2_code_2021', 'LGA_NAME_2025']],
    on='sa2_code_2021',
    how='left'
)

print(f"SA2s unmatched to LGA: {melb_pop['LGA_NAME_2025'].isna().sum()}")
print("\nTop LGAs by SA2 count:")
print(melb_pop['LGA_NAME_2025'].value_counts().head(20).to_string())
print("\nSample pairings:")
print(melb_pop[['sa2_name_2021', 'LGA_NAME_2025']].head(10).to_string(index=False))

## Calculate Population Growth Metrics

We calculate two growth metrics:
1. **Cumulative growth (2021 to 2024)**: Total percentage change over 3 years. This captures accumulated demand pressure.
2. **Single-year growth (2023 to 2024)**: Already in the dataset as `erp_change_per_cent_2023_24`. Shows the most recent trend.

In [ ]:
# Cumulative percentage growth from 2021 to 2024
melb_pop['cumulative_growth_pct'] = (
    (melb_pop['erp_2024'] - melb_pop['erp_2021']) 
    / melb_pop['erp_2021'] * 100
).round(2)

# Absolute population added
melb_pop['population_added_2021_24'] = melb_pop['erp_2024'] - melb_pop['erp_2021']

print("=== TOP 15 FASTEST GROWING SA2s (2021-2024) ===")
print(melb_pop.nlargest(15, 'cumulative_growth_pct')[[
    'sa2_name_2021', 'erp_2021', 'erp_2024',
    'cumulative_growth_pct', 'population_added_2021_24',
    'erp_change_per_cent_2023_24'
]].to_string(index=False))

## Filter and Categorize Infrastructure

The Vicmap FOI dataset contains ~200 feature subtypes covering everything from  
schools to silos to telecommunications towers. We filter to only **social infrastructure**  
— public services that residents actively use — and tag each facility with a  
high-level category for color-coded visualization in Kepler.gl.

**Categories:** Education, Health, Community, Worship, Emergency, Government, Recreation

In [ ]:
infra_categories = {
    # Education
    'primary school': 'Education',
    'secondary school': 'Education',
    'primary/secondary school': 'Education',
    'special school': 'Education',
    'tertiary institution': 'Education',
    'university': 'Education',
    'further education': 'Education',
    'child care': 'Education',
    'kindergarten': 'Education',
    
    # Health
    'general hospital': 'Health',
    'general hospital (emergency)': 'Health',
    'community health centre': 'Health',
    'health centre': 'Health',
    'maternal/child health centre': 'Health',
    'disability support centre': 'Health',
    'aged care': 'Health',
    'day procedure centre': 'Health',
    
    # Community
    'library': 'Community',
    'community centre': 'Community',
    'neighbourhood house': 'Community',
    'senior citizens': 'Community',
    'hall': 'Community',
    'art gallery': 'Community',
    'museum': 'Community',
    
    # Worship
    'church': 'Worship',
    'mosque': 'Worship',
    'monastry': 'Worship',
    'gurdwara (sikh)': 'Worship',
    'vihara (buddhist)': 'Worship',
    'synagogue': 'Worship',
    'mandir (hindu)': 'Worship',
    
    # Emergency
    'police station': 'Emergency',
    'fire station': 'Emergency',
    'ses unit': 'Emergency',
    'ambulance station': 'Emergency',
    'neighbourhood safer place': 'Emergency',
    'lifesaving club': 'Emergency',
    
    # Government
    'post office': 'Government',
    'municipal office': 'Government',
    'customer service centre': 'Government',
    'justice service': 'Government',
    'law court': 'Government',
    
    # Recreation
    'swimming pool': 'Recreation',
    'indoor recreation centre': 'Recreation',
    'sports facility': 'Recreation',
    'playground': 'Recreation',
    'picnic site': 'Recreation',
    'entertainment centre': 'Recreation',
    'club house': 'Recreation'
}

# Filter to social infrastructure only and assign categories
public_infra = infra_gdf[infra_gdf['FEATSUBTYP'].isin(infra_categories.keys())].copy()
public_infra['infra_category'] = public_infra['FEATSUBTYP'].map(infra_categories)
public_infra = public_infra.to_crs(epsg=7844)

print(f"Social infrastructure points (all Victoria): {len(public_infra)}")
print(f"Infrastructure types included: {len(infra_categories)}")
print(f"\nBy category:")
print(public_infra['infra_category'].value_counts().to_string())

## Spatial Clip To Keep Only Melbourne Infrastructure

The Vicmap data covers all of Victoria. We use `gpd.clip()` against the  
dissolved Melbourne boundary to keep only points physically inside  
Greater Melbourne. This prevents regional data (Geelong, Bendigo, etc.)  
from appearing on the map.

In [ ]:
melb_infra = gpd.clip(public_infra, melb_boundary)

print(f"Before clip (all Victoria): {len(public_infra)}")
print(f"After clip (Melbourne only): {len(melb_infra)}")
print(f"Points removed: {len(public_infra) - len(melb_infra)}")

### Verification: Visual Check of Spatial Clip

Left panel shows infrastructure scattered across all of Victoria.  
Right panel should show all dots contained within the Melbourne boundary with no leakage.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

axes[0].set_title("BEFORE Clip (all Victoria)", fontsize=13)
melb_pop.plot(ax=axes[0], facecolor='#f0f0f0', edgecolor='white', linewidth=0.3)
public_infra.plot(ax=axes[0], color='red', markersize=0.5, alpha=0.4)
axes[0].axis('off')

axes[1].set_title("AFTER Clip (Melbourne only)", fontsize=13)
melb_pop.plot(ax=axes[1], facecolor='#f0f0f0', edgecolor='white', linewidth=0.3)
melb_infra.plot(ax=axes[1], color='red', markersize=0.5, alpha=0.4)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Enrich Infrastructure with SA2 Population Context

Each infrastructure point is spatially joined to its containing SA2 polygon.  
This attaches suburb name and population growth data to every facility

In [ ]:
melb_infra_enriched = gpd.sjoin(
    melb_infra,
    melb_pop[['sa2_name_2021', 'LGA_NAME_2025', 'erp_2021', 'erp_2024',
              'erp_change_per_cent_2023_24',
              'cumulative_growth_pct', 'population_added_2021_24', 'geometry']],
    how='left',
    predicate='within'
)

# Clean up join artifacts
melb_infra_enriched = melb_infra_enriched.drop(columns=['index_right'], errors='ignore')

# Remove duplicates from points on polygon edges matching multiple SA2s
before = len(melb_infra_enriched)
melb_infra_enriched = melb_infra_enriched.drop_duplicates(
    subset=['NAME', 'FEATSUBTYP', 'geometry']
)
print(f"Before dedup: {before}")
print(f"After dedup:  {len(melb_infra_enriched)}")
print(f"Unmatched (no SA2): {melb_infra_enriched['sa2_name_2021'].isna().sum()}")

## Calculate Infrastructure Density per SA2

The core analytical metric: **facilities per 1,000 residents**.  
Cross-referencing this against population growth reveals  
"infrastructure poverty" hotspots — high growth suburbs with  
disproportionately low service availability.

In [ ]:
# Count facilities per SA2, broken down by category
infra_count = melb_infra_enriched.groupby('sa2_name_2021').agg(
    total_facilities=('FEATSUBTYP', 'count'),
    education=('infra_category', lambda x: (x == 'Education').sum()),
    health=('infra_category', lambda x: (x == 'Health').sum()),
    community=('infra_category', lambda x: (x == 'Community').sum()),
    worship=('infra_category', lambda x: (x == 'Worship').sum()),
    emergency=('infra_category', lambda x: (x == 'Emergency').sum()),
    government=('infra_category', lambda x: (x == 'Government').sum()),
    recreation=('infra_category', lambda x: (x == 'Recreation').sum()),
).reset_index()

# Merge counts into population data
melb_pop_final = melb_pop.merge(infra_count, on='sa2_name_2021', how='left')

# Fill suburbs with zero infrastructure
fill_cols = ['total_facilities', 'education', 'health', 'community',
             'worship', 'emergency', 'government', 'recreation']
melb_pop_final[fill_cols] = melb_pop_final[fill_cols].fillna(0).astype(int)

# Facilities per 1000 residents
melb_pop_final['facilities_per_1000'] = (
    melb_pop_final['total_facilities'] / melb_pop_final['erp_2024'] * 1000
).round(2)

print("=== INFRASTRUCTURE POVERTY HOTSPOTS ===")
print("(High growth SA2s with fewest facilities per 1000 people)\n")
high_growth = melb_pop_final[melb_pop_final['cumulative_growth_pct'] > 10]
print(high_growth.nsmallest(15, 'facilities_per_1000')[[
    'sa2_name_2021', 'cumulative_growth_pct', 'population_added_2021_24',
    'total_facilities', 'facilities_per_1000'
]].to_string(index=False))

## Parse Infrastructure Creation Dates

The Vicmap `CRDATE_PFI` field records when each facility was first added  
to the database. Pre-2009 facilities were bulk-loaded (dates unreliable),  
but post-2009 dates reflect genuine additions. This enables time-based  
analysis of when infrastructure was built relative to population growth.  

For the Infrastructure Poverty Timeline, those bulk-loaded facilities are retained but later pinned to the  
start of the animation window so the first frame shows existing infrastructure  
stock rather than a misleading 2009 construction spike.

In [ ]:
melb_infra_enriched['facility_created'] = pd.to_datetime(
    melb_infra_enriched['CRDATE_PFI'], errors='coerce'
)
melb_infra_enriched['created_year'] = melb_infra_enriched['facility_created'].dt.year

print("=== MELBOURNE INFRASTRUCTURE BY CREATION YEAR ===")
year_counts = melb_infra_enriched['created_year'].value_counts().sort_index()
print(f"\n2009 (bulk load): {year_counts.get(2009, 0)}")
print("\nPost-bulk-load additions:")
for year in range(2010, 2026):
    count = year_counts.get(year, 0)
    if count > 0:
        print(f"  {year}: {count}")

print(f"\nTotal pre-2021: {(melb_infra_enriched['created_year'] < 2021).sum()}")
print(f"Total 2021 onwards: {(melb_infra_enriched['created_year'] >= 2021).sum()}")

## Clean Data (handle NaN / Infinity)

Some SA2 areas have zero population in 2021 (industrial zones, airports, parks),  
causing division-by-zero in growth calculations. We clean these before export.

In [ ]:
# Replace infinity with NaN, then fill NaN with 0
numeric_cols = melb_pop_final.select_dtypes(include=[np.number]).columns
melb_pop_final[numeric_cols] = melb_pop_final[numeric_cols].replace([np.inf, -np.inf], np.nan)
melb_pop_final[numeric_cols] = melb_pop_final[numeric_cols].fillna(0)

numeric_cols_infra = melb_infra_enriched.select_dtypes(include=[np.number]).columns
melb_infra_enriched[numeric_cols_infra] = melb_infra_enriched[numeric_cols_infra].replace([np.inf, -np.inf], np.nan)
melb_infra_enriched[numeric_cols_infra] = melb_infra_enriched[numeric_cols_infra].fillna(0)

problem_areas = melb_pop[melb_pop['erp_2021'] == 0]['sa2_name_2021'].tolist()
print(f"SA2s with zero 2021 population (caused infinity): {len(problem_areas)}")
for area in problem_areas:
    print(f"  {area}")

## Export — Current Infrastructure Poverty (Static Map)

Two GeoJSON files for the primary Kepler.gl visualization.

In [ ]:
print("Final SA2 columns:")
print(melb_pop_final.columns.tolist())

rename_map = {
    'sa2_name_2021': 'Suburb',
    'LGA_NAME_2025': 'Council',
    'cumulative_growth_pct': 'Population growth 2021 to 2024 (%)',
    'facilities_per_1000': 'Facilities per 1000 residents',
    'pop_density_2024_people_per_km2': 'Population density (people per km2)',
    'erp_2021': 'Population 2021',
    'erp_2024': 'Population 2024',
}

export_gdf = melb_pop_final.rename(columns=rename_map)

print("\nExport columns (renamed):")
print(export_gdf.columns.tolist())

export_gdf.to_file(POPULATION_GROWTH_OUTPUT, driver='GeoJSON')
print(f"\nExported: {POPULATION_GROWTH_OUTPUT} ({len(export_gdf)} SA2 areas)")

melb_infra_enriched.to_file(INFRA_POINTS_OUTPUT, driver='GeoJSON')
print(f"Exported: {INFRA_POINTS_OUTPUT} ({len(melb_infra_enriched)} points)")

## Export — Infrastructure Poverty Timeline (Animated Version)

Two additional GeoJSON files for the time-animated visualization.

Infrastructure Poverty Timeline uses a **2010 to 2024** time window. This avoids the misleading  
2009 bulk-load spike in the Vicmap creation dates while keeping  
pre-2010 facilities visible from the first frame.

### Build Population Timeseries

We reshape the SA2 population data into one row per suburb per year and  
recompute infrastructure counts from the Infrastructure Poverty Timeline point timeline as at `30 June`  
of each year. This makes `total_facilities`, category counts, and  
`facilities_per_1000` genuinely time-varying rather than fixed 2024 values.  

Each row receives a mid-year timestamp (`30 June`) because ABS ERP values  
represent end-of-financial-year population estimates.

In [ ]:
# Build a timeline-specific infrastructure copy so Current Infrastructure Poverty metrics stay unchanged.
timeline_start_year = 2010
timeline_end_year = 2024
timeline_start = pd.Timestamp(f'{timeline_start_year}-01-01')
timeline_end = pd.Timestamp(f'{timeline_end_year}-12-31')
erp_year_map = {year: f'erp_{year}' for year in range(timeline_start_year, timeline_end_year + 1)}

melb_infra_timeline = melb_infra_enriched.copy()
melb_infra_timeline['timestamp'] = pd.to_datetime(melb_infra_timeline['CRDATE_PFI'], errors='coerce')

before = len(melb_infra_timeline)
melb_infra_timeline = melb_infra_timeline.dropna(subset=['timestamp']).copy()
rows_dropped_infra_ts_invalid = before - len(melb_infra_timeline)

rows_clamped_infra_ts_before_2010 = (melb_infra_timeline['timestamp'] < timeline_start).sum()
melb_infra_timeline.loc[melb_infra_timeline['timestamp'] < timeline_start, 'timestamp'] = timeline_start
before_end_filter = len(melb_infra_timeline)
melb_infra_timeline = melb_infra_timeline[melb_infra_timeline['timestamp'] <= timeline_end].copy()
rows_dropped_infra_ts_after_2024 = before_end_filter - len(melb_infra_timeline)
melb_infra_timeline['existed_by'] = melb_infra_timeline['timestamp'].dt.strftime('%Y-%m-%d')

timeline_count_cols = [
    'total_facilities', 'education', 'health', 'community',
    'worship', 'emergency', 'government', 'recreation'
]

pop_rows = []
for year, col in erp_year_map.items():
    year_slice = melb_pop_final[[
        'sa2_name_2021', 'LGA_NAME_2025', col, 'cumulative_growth_pct', 'geometry'
    ]].copy()
    year_slice = year_slice.rename(columns={col: 'population'})
    year_slice['year'] = year
    year_slice['baseline_population_2021'] = melb_pop_final['erp_2021'].values

    year_timestamp = pd.Timestamp(f'{year}-06-30')
    infra_existing = melb_infra_timeline[melb_infra_timeline['timestamp'] <= year_timestamp]

    infra_count_year = infra_existing.groupby('sa2_name_2021').agg(
        total_facilities=('FEATSUBTYP', 'count'),
        education=('infra_category', lambda x: (x == 'Education').sum()),
        health=('infra_category', lambda x: (x == 'Health').sum()),
        community=('infra_category', lambda x: (x == 'Community').sum()),
        worship=('infra_category', lambda x: (x == 'Worship').sum()),
        emergency=('infra_category', lambda x: (x == 'Emergency').sum()),
        government=('infra_category', lambda x: (x == 'Government').sum()),
        recreation=('infra_category', lambda x: (x == 'Recreation').sum()),
    ).reset_index()

    year_slice = year_slice.merge(infra_count_year, on='sa2_name_2021', how='left')
    year_slice[timeline_count_cols] = year_slice[timeline_count_cols].fillna(0).astype(int)
    year_slice['facilities_per_1000'] = (
        year_slice['total_facilities'] / year_slice['population'].replace(0, np.nan) * 1000
    ).round(2).fillna(0)

    # Keep 2021 comparison metrics for consistency with Current Infrastructure Poverty.
    year_slice['growth_from_2021_pct'] = (
        (year_slice['population'] - year_slice['baseline_population_2021'])
        / year_slice['baseline_population_2021'].replace(0, np.nan) * 100
    ).round(2).fillna(0)
    year_slice['people_added_since_2021'] = (
        year_slice['population'] - year_slice['baseline_population_2021']
    )
    year_slice = year_slice.drop(columns=['baseline_population_2021'])

    pop_rows.append(year_slice)

pop_timeseries = gpd.GeoDataFrame(
    pd.concat(pop_rows, ignore_index=True),
    geometry='geometry',
    crs='EPSG:7844'
)

pop_timeseries['timestamp'] = pd.to_datetime(pop_timeseries['year'].astype(str) + '-06-30')

# Get each SA2's 2010 population as the baseline
baseline_2010 = pop_timeseries[pop_timeseries['year'] == 2010].set_index('sa2_name_2021')['population']

# Compute growth since 2010 as % and absolute people added
pop_timeseries['population_2010'] = pop_timeseries['sa2_name_2021'].map(baseline_2010)
pop_timeseries['growth_from_2010_pct'] = (
    (pop_timeseries['population'] - pop_timeseries['population_2010'])
    / pop_timeseries['population_2010'].replace(0, np.nan) * 100
).round(2).fillna(0)
pop_timeseries['people_added_since_2010'] = (
    pop_timeseries['population'] - pop_timeseries['population_2010']
).round(0).astype('Int64')

# Drop the intermediate baseline column, we don't need it in the export
pop_timeseries = pop_timeseries.drop(columns=['population_2010'])

print("\nGrowth check (Tarneit - Central):")
print(
    pop_timeseries[pop_timeseries['sa2_name_2021'] == 'Tarneit - Central'][[
        'year', 'population', 'growth_from_2010_pct', 'people_added_since_2010'
    ]].sort_values('year').to_string(index=False)
)

export_pop_ts = pop_timeseries.drop(
    columns=['growth_from_2021_pct', 'people_added_since_2021'],
    errors='ignore'
).rename(columns={
    'sa2_name_2021': 'Suburb',
    'LGA_NAME_2025': 'Council',
    'population': 'Population',
    'total_facilities': 'Total facilities',
    'facilities_per_1000': 'Facilities per 1000 residents',
    'cumulative_growth_pct': 'Cumulative growth 2021 to 2024 (%)',
    'growth_from_2010_pct': 'Growth since 2010 (%)',
    'people_added_since_2010': 'People added since 2010',
    'education': 'Education facilities',
    'health': 'Health facilities',
    'community': 'Community facilities',
    'worship': 'Worship facilities',
    'emergency': 'Emergency facilities',
    'government': 'Government facilities',
    'recreation': 'Recreation facilities',
})

export_pop_ts.to_file(POPULATION_TS_OUTPUT, driver='GeoJSON')
print(f"Exported: {POPULATION_TS_OUTPUT}")
print(f"  {len(melb_pop_final)} SA2s x {len(erp_year_map)} years = {len(export_pop_ts)} rows")
print(f"  Population years exported: {timeline_start_year}-{timeline_end_year}")
print(f"  Timestamp range: {export_pop_ts['timestamp'].min().date()} to {export_pop_ts['timestamp'].max().date()}")
print("\nCurrent Infrastructure Poverty SA2 columns:")
print(melb_pop_final.columns.tolist())
print("\npop_timeseries columns:")
print(pop_timeseries.columns.tolist())
print("\nexport_pop_ts columns:")
print(export_pop_ts.columns.tolist())

### Build Infrastructure Timeseries

This export uses the same Infrastructure Poverty Timeline-ready infrastructure copy used to build the  
population timeseries counts. Rows with invalid dates are removed, dates after  
2024 are excluded, and pre-2010 facilities are pinned to `2010-01-01` so they  
appear in the opening frame without being presented as real 2009 additions.  

The original `created_year` is kept in the export for reference.

In [ ]:
print(f"Dropped {rows_dropped_infra_ts_invalid} rows with unparseable CRDATE_PFI")
print(f"Clamped {rows_clamped_infra_ts_before_2010} rows before 2010-01-01 up to 2010-01-01 for the Infrastructure Poverty Timeline")
print(f"Dropped {rows_dropped_infra_ts_after_2024} rows after 2024-12-31 for the Infrastructure Poverty Timeline")
print(melb_infra_timeline['timestamp'].describe())

export_infra_ts = melb_infra_timeline.rename(columns={
    'NAME': 'Facility name',
    'FEATSUBTYP': 'Facility type',
    'infra_category': 'Category',
    'sa2_name_2021': 'Suburb',
    'LGA_NAME_2025': 'Council',
    'cumulative_growth_pct': 'Suburb growth 2021 to 2024 (%)',
})

print("\nmelb_infra_timeline columns:")
print(melb_infra_timeline.columns.tolist())
print("\nexport_infra_ts columns:")
print(export_infra_ts.columns.tolist())
print(f"\nRows dropped from infra timeseries (invalid dates): {rows_dropped_infra_ts_invalid}")
print(f"Rows clamped to 2010-01-01 in infra timeseries: {rows_clamped_infra_ts_before_2010}")
print(f"Rows dropped from infra timeseries (after 2024): {rows_dropped_infra_ts_after_2024}")
print("\nInfrastructure timestamp range:")
print(melb_infra_timeline['timestamp'].describe())

export_infra_ts.to_file(INFRA_TS_OUTPUT, driver='GeoJSON')
print(f"\nExported: {INFRA_TS_OUTPUT} ({len(export_infra_ts)} points)")

print("\n=== WHAT THE TIME FILTER SHOWS ===")
for year in [2021, 2022, 2023, 2024]:
    year_mid = pd.Timestamp(f'{year}-06-30')
    year_start = pd.Timestamp(f'{year}-01-01')
    year_end = pd.Timestamp(f'{year}-12-31')
    count = (melb_infra_timeline['timestamp'] <= year_mid).sum()
    new_that_year = ((melb_infra_timeline['timestamp'] >= year_start) & (melb_infra_timeline['timestamp'] <= year_end)).sum()
    print(f"  By mid-{year}: {count} facilities existed (+{new_that_year} new that year)")

## Export Summary CSV

Flat CSV of all SA2 areas with growth metrics and facility counts,  
sorted by cumulative growth. Useful for tables, charts, and presentations.

In [ ]:
summary = melb_pop_final[[
    'sa2_name_2021', 'LGA_NAME_2025', 'erp_2021', 'erp_2024',
    'pop_density_2024_people_per_km2',
    'cumulative_growth_pct', 'erp_change_per_cent_2023_24',
    'population_added_2021_24', 'total_facilities',
    'education', 'health', 'community', 'worship',
    'emergency', 'government', 'recreation', 'facilities_per_1000'
]].sort_values('cumulative_growth_pct', ascending=False)

summary = summary.rename(columns={
    'sa2_name_2021': 'Suburb',
    'LGA_NAME_2025': 'Council',
    'cumulative_growth_pct': 'Population growth 2021 to 2024 (%)',
    'facilities_per_1000': 'Facilities per 1000 residents',
    'pop_density_2024_people_per_km2': 'Population density (people per km2)',
    'erp_2021': 'Population 2021',
    'erp_2024': 'Population 2024',
})

summary.to_csv(SUMMARY_OUTPUT, index=False)
print(f"Exported: {SUMMARY_OUTPUT} ({len(summary)} rows)")
print(f"Columns: {summary.columns.tolist()}")

## Infrastructure Type Included/Excluded

Shows which Vicmap feature subtypes are included vs excluded.  
Included: social infrastructure that provides active services to residents.  
Excluded: industrial, telecommunications, tourism, and non-residential types.

In [ ]:
print("=== INCLUDED TYPES ===")
used = melb_infra_enriched['FEATSUBTYP'].value_counts()
print(f"Types: {len(used)} | Total points: {len(melb_infra_enriched)}\n")
print(used.to_string())

print("\n\n=== EXCLUDED TYPES ===")
all_types = set(infra_gdf['FEATSUBTYP'].unique())
used_types = set(infra_categories.keys())
excluded = all_types - used_types
print(f"Types: {len(excluded)}\n")
for t in sorted(excluded):
    count = len(infra_gdf[infra_gdf['FEATSUBTYP'] == t])
    print(f"  {t}: {count}")

---

## Output Files

| File | Purpose | Kepler Usage |
|---|---|---|
| `outputs/geojson/kepler_population_growth.geojson` | Current Infrastructure Poverty: SA2 polygons with growth + facility counts | Choropleth base layer |
| `outputs/geojson/kepler_infrastructure_points.geojson` | Current Infrastructure Poverty: Individual facilities with category + SA2 context | Point overlay |
| `outputs/geojson/kepler_population_timeseries.geojson` | Infrastructure Poverty Timeline: Year-by-year SA2 population snapshots (2010-2024) | Time-animated choropleth |
| `outputs/geojson/kepler_infrastructure_timeseries.geojson` | Infrastructure Poverty Timeline: Facilities with timestamps aligned to a 2010-2024 animation window | Time-filtered points |
| `outputs/tables/infrastructure_poverty_summary.csv` | All SA2 areas ranked by growth and facility density | Tables and charts |